In [1]:
import os
import random
import sys
import warnings
from pathlib import Path
from __future__ import annotations

import numpy as np
import pandas as pd
import optuna
import time

PROJECT_ROOT = Path.cwd().parent
SRC_PATH = PROJECT_ROOT / "src"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

from utils import eda_toolkit as toolkit
from utils import build_features as bf
from utils import trainer_toolkit as tt
from utils import test_toolkit as ts

from eval.scorer import DemandScorer
from eval.holdout import build_holdout_dataset, evaluate_on_holdout
from models.baselines import seasonal_naive, historical_median
from models.lgbm_tweedie_wrapper import LightGBMTweedieWrapper
from trainer.walk_forward import WalkForwardTrainer

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

RESULTS_PATH = PROJECT_ROOT / "results" / "lgbm_k54"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

train = pd.read_parquet(DATA_PROCESSED / "train_k54.parquet")
holdout = pd.read_parquet(DATA_PROCESSED / "holdout_k54.parquet")

feature_builder = lambda df: bf.build_features(df, dropna=True)

assert train.index.max() < holdout.index.min(), "Solapamiento temporal train/holdout"

print("=" * 70)
print(f"Train:   {len(train):>6} filas  ({train.index[0]} -> {train.index[-1]})")
print(f"Holdout: {len(holdout):>6} filas  ({holdout.index[0]} -> {holdout.index[-1]})")
print("=" * 70)

Train:    56992 filas  (2023-01-01 04:30:00 -> 2025-05-26 17:15:00)
Holdout:  14248 filas  (2025-05-26 17:30:00 -> 2025-12-31 20:30:00)


# Introducción

El Gradient Boosting con LightGBM es un marco de aprendizaje supervisado que construye un ensamble aditivo de árboles de decisión mediante la optimización secuencial de una función de pérdida diferenciable. A diferencia de los modelos que imponen una ecuación paramétrica para la evolución de la media condicional, LightGBM aprende la relación entre las variables predictoras y la respuesta directamente a partir de los datos, sin asumir una forma funcional específica.

En el contexto de la demanda de pasajeros por franja horaria, esta arquitectura ofrece una alternativa a los modelos recurrentes y a los enfoques econométricos tradicionales, al permitir que la relación entre el calendario, la estacionalidad, los rezagos y la demanda futura sea aprendida directamente de los datos. La naturaleza de la respuesta, sin embargo, impone una restricción sobre la función de pérdida: la serie presenta sobredispersión marcada en las horas de mayor operación y una proporción apreciable de franjas con cero pasajeros, estructura que una pérdida cuadrática o una verosimilitud Poisson subestimarían. Por esta razón se adopta la distribución Tweedie como familia de verosimilitud del objetivo, cuya forma compuesta Poisson-Gamma asigna masa puntual en cero y una densidad continua para los valores positivos.

El objetivo de esta sección es presentar, entrenar y evaluar modelos LightGBM con pérdida Tweedie para la predicción de la demanda de pasajeros por franja horaria en la ruta K54. Dado que la operación requiere pronósticos a múltiples escalas temporales, el problema se formula como una tarea de predicción directa multi-horizonte. Para cada intervalo operativo solicitado (30, 60 y 120 minutos) se entrena y optimiza un modelo LightGBM independiente, cada uno con su propio objetivo agregado y su propia validación. La evaluación sobre el *holdout* replica las condiciones de despliegue operativo mediante un reentrenamiento rodante por bloques con ventana deslizante. Este esquema respeta la partición cronológica y las restricciones metodológicas del proyecto: emplea exclusivamente variables disponibles en el instante de decisión y previene cualquier fuga de información entre conjuntos.

### Arquitectura

LightGBM construye un modelo aditivo de árboles de decisión que aproxima la función objetivo de forma secuencial. Para un conjunto de $N$ observaciones $\{\mathbf{x}_i, y_i\}_{i=1}^{N}$, el modelo final se expresa como la suma de $M$ árboles $h_m(\cdot)$ escalados por una tasa de aprendizaje $\nu$:

$$
F_M(\mathbf{x}_i) = \sum_{m=1}^{M} \nu \, h_m(\mathbf{x}_i),
$$

donde cada árbol $h_m \in \mathcal{H}$ es una función escalonada sobre el espacio de predictores. El ajuste es *forward stagewise*: en la iteración $m$, el modelo previo $F_{m-1}$ se mantiene fijo y se añade un nuevo árbol que corrige sus errores,

$$
F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \nu \, h_m(\mathbf{x}).
$$

El nuevo árbol se obtiene minimizando una expansión de segundo orden de la pérdida $\mathcal{L}$ alrededor de la predicción actual. Definiendo el gradiente $g_{im}$ y el hessiano $h_{im}$ de la pérdida respecto a $F_{m-1}(\mathbf{x}_i)$,

$$
g_{im} = \left. \frac{\partial \mathcal{L}(y_i, F)}{\partial F} \right|_{F = F_{m-1}(\mathbf{x}_i)}, \qquad
h_{im} = \left. \frac{\partial^2 \mathcal{L}(y_i, F)}{\partial F^2} \right|_{F = F_{m-1}(\mathbf{x}_i)},
$$

las hojas del árbol se calculan con la solución de Newton,

$$
w_{j}^{*} = - \frac{\sum_{i \in I_j} g_{im}}{\sum_{i \in I_j} h_{im} + \lambda},
$$

donde $I_j$ es el conjunto de observaciones que caen en la hoja $j$ y $\lambda$ es el término de regularización $\ell_2$. La ganancia de una partición que divide $I$ en $I_L$ e $I_R$ se evalúa como:

$$
\mathcal{G} = \frac{1}{2}\left[
\frac{\left(\sum_{i \in I_L} g_{im}\right)^2}{\sum_{i \in I_L} h_{im} + \lambda} +
\frac{\left(\sum_{i \in I_R} g_{im}\right)^2}{\sum_{i \in I_R} h_{im} + \lambda} -
\frac{\left(\sum_{i \in I} g_{im}\right)^2}{\sum_{i \in I} h_{im} + \lambda}
\right].
$$

Bajo el objetivo `tweedie`, LightGBM asume que la respuesta sigue una distribución Tweedie con parámetro de potencia $p$ y utiliza una función de enlace logarítmica, de modo que la media condicional se obtiene como $\hat{\mu}(\mathbf{x}) = \exp(F(\mathbf{x}))$. La pérdida que se minimiza es la desviación negativa de la log-verosimilitud Tweedie,

$$
\mathcal{L}_{\text{Tweedie}}(y, \hat{\mu}) = - \left[ y \, \frac{\hat{\mu}^{1-p}}{1-p} - \frac{\hat{\mu}^{2-p}}{2-p} \right].
$$

:::{admonition} Información: Distribución Tweedie
:class: info

La distribución Tweedie con parámetro de potencia $p$ pertenece a la familia exponencial y se caracteriza por la relación entre la media $\mu$ y la varianza:

$$
\operatorname{Var}(y) = \phi \, \mu^{p}, \qquad 1 < p < 2.
$$

Valores de $p$ cercanos a 1 aproximan la verosimilitud a una Poisson; valores cercanos a 2 la aproximan a una Gamma; los valores intermedios corresponden a la familia compuesta Poisson-Gamma, que admite masa en cero y una componente continua positiva. En este proyecto, el parámetro $p$ se estima empíricamente a partir de la relación varianza-media de la serie. El valor resultante se fija como constante durante el entrenamiento para garantizar la estabilidad y comparabilidad de la métrica.
:::

Con la función de enlace logarítmica, las derivadas de la pérdida respecto a la predicción del modelo son:

$$
\frac{\partial \mathcal{L}}{\partial F} = -y \, e^{(1-p)F} + e^{(2-p)F}, \qquad
\frac{\partial^2 \mathcal{L}}{\partial F^2} = -y(1-p) \, e^{(1-p)F} + (2-p) \, e^{(2-p)F}.
$$

Estas expresiones alimentan el gradiente $g_{im}$ y el hessiano $h_{im}$ en cada iteración del boosting y son las que permiten que el modelo pondere de forma diferenciada las franjas con cero pasajeros y las franjas de alta demanda. Para $1 < p < 2$, el hessiano es estrictamente positivo, lo que garantiza la convexidad local del subproblema de optimización en cada hoja.

El modelo completo $F_M(\cdot)$ asigna a cada vector de predictores $\mathbf{x}$ (hora, día de la semana, mes, codificación cíclica, estacionalidad, rezagos y tendencia temporal) una predicción de la media condicional de pasajeros acumulados en la ventana objetivo, $\hat{\mu}(\mathbf{x}) = \exp(F_M(\mathbf{x}))$.

### Protocolo de entrenamiento y validación

Se entrena un modelo LightGBM independiente por cada horizonte de predicción (30, 60 y 120 minutos, equivalentes a $h \in \{2, 4, 8\}$ franjas). El *target* de cada modelo es la suma rodante de pasajeros desde el instante de decisión hasta el final del intervalo, compartiendo todos el mismo conjunto de 23 variables predictoras. Dado que la agregación temporal altera la dispersión de la demanda, el parámetro de potencia $p$ de Tweedie se estima empíricamente de forma independiente para cada horizonte y se fija como constante durante su entrenamiento. Cada intervalo se predice en un único paso, sin usar las predicciones de un horizonte como insumo de otro.

Para mitigar el cambio de régimen estructural entre 2023 y 2025 y evitar sesgos por sobrepredicción, se implementó una ventana deslizante (*sliding window*). El hiperparámetro `window` define la historia reciente que observa el modelo; se exploraron tamaños entre 18,000 y 26,000 franjas (de 8 a 12 meses), con un calentamiento inicial fijado en 26,000.

La validación en `train.parquet` usa un esquema *walk-forward* de 5 pliegues, aplicando un *embargo* de $h-1$ franjas en la frontera para evitar fugas de información. Para la evaluación final, el `holdout.parquet` replica esta mecánica avanzando cronológicamente y reentrenando el modelo por bloques con las últimas `window` franjas disponibles.

La implementación utiliza `LightGBM` (v4.6) con el objetivo `tweedie`. El *early stopping* interno se evalúa sobre una partición cronológica 70/30 dentro de cada bloque de entrenamiento, respetando también el *embargo* del horizonte.

## Ventana de predicción de 30 minutos

### Entrenamiento

Para determinar el parámetro de potencia $p$ en la muestra de entrenamiento, se aplicó un estimador por momentos basado en la relación teórica entre la varianza y la media de la distribución Tweedie:

$$
\log(\operatorname{Var}(y)) = \log(\phi) + p \log(\mu)
$$

Al agrupar las observaciones por hora y día de la semana, y calcular la pendiente de esta regresión lineal sobre las medias y varianzas resultantes, se obtuvo un valor de $p=1.4$. Al contrastar este ajuste con los datos reales, el análisis visual revela que la curva teórica no alcanza la altura máxima del pico empírico y presenta desviaciones en los extremos del Q-Q plot. Una hipótesis plausible para explicar este comportamiento es la alta heterogeneidad de la serie: al intentar abarcar la gran dispersión generada por la mezcla de regímenes operativos (horas pico, valles y fines de semana), la distribución teórica se aplana en el centro y engrosa sus colas. Dicho sesgo marginal no afecta el desempeño predictivo del modelo LightGBM, ya que la función de pérdida Tweedie se aplica de forma condicional sobre las hojas del árbol, capturando la dinámica local de cada franja en lugar del promedio global.

In [2]:
warmup_data = train["pasajeros"].iloc[:26000]
pasajeros_h2_warmup = warmup_data.rolling(window=2).sum().shift(-1).dropna()

p_warmup = tt._estimate_tweedie_p_internal(pasajeros_h2_warmup)

_, _ = tt.plot_tweedie_fit(
    series=pasajeros_h2_warmup,
    p=p_warmup,
    titles=("Distribución de Pasajeros (Warmup)", "Q-Q Plot (Warmup)"),
    xlab="Pasajeros acumulados en 30 min (h=2)",
    pads=(15, 15, 12),
    legend_bbox=(1.0, 0.95)
)

Para la selección de hiperparámetros del modelo LightGBM correspondiente al horizonte de 30 minutos ($h = 2$ franjas), se implementó una validación cruzada temporal de 5 pliegues sobre el conjunto de entrenamiento. La búsqueda de Optuna exploró la complejidad de los árboles de decisión, los parámetros de regularización y el tamaño de la ventana de historia reciente.

El modelo recibe directamente una matriz tabular con las 23 variables predictoras (calendario, codificación cíclica, estacionalidad, tendencia temporal y rezagos precalculados). Para evitar el sesgo de anticipación (*look-ahead bias*) y adaptarse al cambio de régimen estructural de la demanda, se implementó una estrategia de ventana deslizante (*sliding window*). En cada pliegue, el modelo se reentrena con un bloque de historia delimitado por el hiperparámetro `window` y se valida con el bloque cronológico inmediatamente posterior, aplicando un *embargo* de una franja ($h - 1 = 1$) para garantizar la integridad temporal de los datos.

El espacio de búsqueda incluyó el tamaño de la ventana de entrenamiento (`window`), el número máximo de hojas por árbol (`num_leaves`), la tasa de aprendizaje (`learning_rate`), el mínimo de datos y de suma de hessianos por hoja (`min_data_in_leaf`, `min_sum_hessian_in_leaf`), las fracciones y frecuencia de submuestreo (`bagging_fraction`, `bagging_freq`, `feature_fraction`) y las penalizaciones por regularización $\ell_1$ y $\ell_2$. El entrenamiento se configuró con un máximo de 5,000 árboles y una paciencia de 300 iteraciones para la parada temprana (*early stopping*) interna, la cual se evalúa sobre el último 10% cronológico de cada bloque de entrenamiento, respetando también el embargo de una franja. El desempeño se evaluó mediante la desviación Tweedie, fijando el parámetro de potencia en $p = p=1.345891937980703$ (estimado exclusivamente con el bloque de calentamiento para evitar fuga de información) y aplicando un límite superior de 100 por observación para mitigar el efecto de eventos atípicos impredecibles.

In [3]:
WINDOW_MAX = 26000

def lgbm_search_space(trial: optuna.Trial) -> dict:
    return {
        "window": trial.suggest_categorical(
            "window", [18000, 20000, 22000, 24000, WINDOW_MAX]
        ),
        "num_leaves": trial.suggest_int("num_leaves", 15, 300, log=True),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.2, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 10, 500, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0, step=0.05),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0, step=0.05),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 50.0, log=True),
        "min_sum_hessian_in_leaf": trial.suggest_float(
            "min_sum_hessian_in_leaf", 1e-3, 10.0, log=True
        ),
    }

exec_id = 30
exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=exec_dir,
    feature_builder=feature_builder,
    horizon_slots=2,
    n_splits=5,
    warmup=WINDOW_MAX,
    embargo=1,
    n_trials=100,
    suggest_params_fn=lgbm_search_space,
    early_stop_rounds=30,
    inner_val_frac=0.1,
    model_kwargs={
        "tweedie_variance_power": 1.345891937980703,
        "n_estimators": 5000,
        "early_stopping_rounds": 300,
        "num_threads": 8,
        "seed": 42,
    },
)

study = trainer.run(train)
best_params = dict(study.best_params)

  0%|          | 0/100 [00:00<?, ?it/s]

Respecto a la eficiencia del algoritmo de búsqueda, la optimización ha completado 66 de los 100 ensayos programados, sin que hasta el momento se haya activado el mecanismo de parada temprana. La mejor configuración se identificó en el ensayo 35, alcanzando una Desviación Tweedie de 4.96142. Este resultado evidencia que el algoritmo ha continuado explorando el espacio de hiperparámetros durante los 30 ensayos posteriores sin encontrar una configuración que supere el desempeño obtenido, aunque todavía resta completar el presupuesto de búsqueda establecido.

La ejecución parcial del proceso de optimización acumula una duración aproximada de 5 minutos y 30 segundos (330 segundos), equivalente a un promedio de 5.00 segundos por ensayo, con un tiempo restante estimado de 3 minutos y 41 segundos. Este comportamiento refleja un costo computacional relativamente reducido para la exploración de las 66 configuraciones evaluadas. 

In [4]:
resumen_data = []

exec_id = 30
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_tweedie = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(
            study_name="LightGBMTweedieWrapper_h2_study", 
            storage=storage_url
        )

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])
        
        mejor_tweedie = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Fallidos": fallidos,
    "Mejor Tweedie Dev": mejor_tweedie,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Fallidos,Mejor Tweedie Dev,Tiempo Total (s),Tiempo por Fold (s)
66,56,10,0,4.9614,269.2800,0.8440


Respecto a la eficiencia del proceso de optimización, se ejecutaron 66 ensayos, de los cuales 56 finalizaron satisfactoriamente, 10 fueron descartados mediante el mecanismo de poda (*pruning*) y ninguno presentó fallos durante su ejecución. La mejor configuración alcanzó una Desviación Tweedie de 4.9614, correspondiente al menor valor de la función objetivo identificado hasta ese momento. La intervención del mecanismo de poda representó aproximadamente el 15.15% de los ensayos ejecutados, mientras que el 84.85% restante completó el procedimiento de evaluación, evidenciando la capacidad del algoritmo para descartar anticipadamente configuraciones con bajo desempeño predictivo.

El entrenamiento efectivo de los modelos a través de los 280 pliegues correspondientes a los ensayos completados (56 ensayos × 5 pliegues) acumuló un tiempo neto de 269.28 segundos (4.49 minutos), equivalente a un promedio de 0.8440 segundos por pliegue bajo el esquema de ventana deslizante. Este resultado refleja un costo computacional reducido durante la evaluación de las configuraciones.

In [5]:
exec_dir = RESULTS_PATH / "exec_30"
df_lgbm = pd.read_csv(exec_dir / "LightGBMTweedieWrapper_results.csv")

df_lgbm["lr"] = df_lgbm["learning_rate"].apply(lambda x: f"{x:.4f}")
df_lgbm["l1"] = df_lgbm["lambda_l1"].apply(lambda x: f"{x:.1e}")
df_lgbm["l2"] = df_lgbm["lambda_l2"].apply(lambda x: f"{x:.1e}")

cols_agrupar = [
    "window", 
    "num_leaves", 
    "min_data_in_leaf", 
    "feature_fraction", 
    "bagging_fraction", 
    "lr", 
    "l1",
    "l2"
]

df_agg = df_lgbm.groupby(cols_agrupar)["tweedie_deviance"].agg(["mean", "std"]).reset_index()

df_agg["Tweedie Deviance"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)

df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "window": "Ventana",
    "num_leaves": "Hojas",
    "min_data_in_leaf": "Min Data Hoja",
    "feature_fraction": "Feat. Frac",
    "bagging_fraction": "Bagging Frac",
    "lr": "LR",
    "l1": "L1",
    "l2": "L2"
})

df_final = df_top5[
    ["Ventana", "Hojas", "Min Data Hoja", "Feat. Frac", "Bagging Frac", "LR", "L1", "L2", "Tweedie Deviance"]
]

tt.display_styled_df(df=df_final, decimals=4, align_left_first=False)

Ventana,Hojas,Min Data Hoja,Feat. Frac,Bagging Frac,LR,L1,L2,Tweedie Deviance
"26,000",66,98,0.5500,0.7500,0.0533,1.1e-08,2.3e-04,4.961 ± 1.539
"26,000",46,49,0.4000,0.8000,0.0184,2.0e-04,1.9e-03,4.976 ± 1.478
"26,000",84,100,0.4500,0.6500,0.0152,4.5e-08,1.1e-04,4.982 ± 1.572
"26,000",96,67,0.4500,0.6000,0.0110,1.7e-07,4.3e-04,4.982 ± 1.532
"26,000",108,49,0.4500,0.6500,0.0308,3.5e-07,2.8e-04,4.989 ± 1.464


La selección del modelo ganador entre las cinco mejores configuraciones se resolvió a favor de la primera (Desviación Tweedie de 4.961). Además de obtener el menor promedio de pérdida, aunque con una variabilidad entre pliegues ligeramente superior a la segunda opción ($\sigma = 1.539$ frente a $\sigma = 1.478$), el modelo seleccionado presenta una mayor capacidad expresiva mediante árboles de 66 hojas frente a las 46 de su competidor más cercano. Esta complejidad estructural se acompaña de un mínimo de 98 observaciones por hoja, un submuestreo moderado de predictores (`feature_fraction` = 0.55) y una fracción de muestreo de observaciones del 75% (`bagging_fraction` = 0.75), que contribuyen a regularizar el ensamble. Asimismo, emplea una tasa de aprendizaje superior (0.0533 frente a 0.0184) y una penalización $\ell_1$ prácticamente nula ($1.1\times10^{-8}$), junto con una regularización $\ell_2$ reducida ($2.3\times10^{-4}$), favoreciendo una actualización relativamente rápida del ensamble sin imponer restricciones significativas sobre los pesos de las hojas.

In [7]:
best_params = {
    "window": 26000,
    "num_leaves": 66,
    "learning_rate": 0.0533,
    "min_data_in_leaf": 98,
    "feature_fraction": 0.55,
    "bagging_fraction": 0.75,
    "bagging_freq": 2,
    "lambda_l1": 1.1e-08,
    "lambda_l2": 2.3e-04,
    "min_sum_hessian_in_leaf": 0.9347338363641041,
}

window = best_params["window"]
model_params = {k: v for k, v in best_params.items() if k != "window"}

final_kwargs = {
    "tweedie_variance_power": 1.345891937980703,
    "n_estimators": 5000,
    "early_stopping_rounds": 300,
    "inner_val_frac": 0.3,
    "num_threads": 8,
    "seed": 42,
}

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=RESULTS_PATH / "exec_30",
    feature_builder=feature_builder,
    horizon_slots=2,
    n_splits=5,
    warmup=26000,
    embargo=1,
    n_trials=100,
    model_kwargs={},
)

X_train_full, y_train_full, _, _ = trainer._build_dataset(train)

model_production = LightGBMTweedieWrapper(**model_params, **final_kwargs)
model_production.fit(X_train_full.iloc[-window:], y_train_full[-window:])

Para evaluar la estructura de los residuos tanto en el conjunto de entrenamiento (*In-Sample*) como en el de prueba (*Out-of-Sample*), a partir de las demandas predichas y las observaciones reales, se calcularon los residuos y su versión estandarizada mediante las siguientes expresiones:

$$
e_t = y_t - \hat{y}_t, \qquad z_t = \frac{e_t - \bar{e}}{s_e}
$$

donde $y_t$ es la demanda observada, $\hat{y}_t$ es la predicción del modelo, $\bar{e}$ es la media muestral de los residuos y $s_e$ es su desviación estándar.

Debido a que los estadísticos de Ljung-Box y ARCH-LM escalan linealmente con el tamaño de la muestra, evaluar la totalidad de la ventana activa de entrenamiento ($26000$ franjas) introduce un efecto de sobre-potencia (*over-power*) que infla los estadísticos ante cualquier dependencia residual. Para atenuar este sesgo sin alterar la secuencia cronológica de los rezagos, el diagnóstico se realizó sobre el último bloque continuo de entrenamiento ($N = 1000$ franjas). A pesar de esta reducción muestral, las pruebas de Ljung-Box (autocorrelación) y ARCH-LM (heterocedasticidad) rechazaron sus hipótesis nulas en todos los rezagos evaluados ($p < .001$).

In [8]:
preds_train_final = model_production.predict_in_sample(X_train_full[-window:])
residuals_train = y_train_full[-window:] - preds_train_final

block_size = 1000
residuals_block = residuals_train[-block_size:]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_train = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_train,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),304.339,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),330.596,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),354.983,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),100.673,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),102.752,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),119.596,< .001,< .001,Efectos ARCH Significativos


La persistencia de autocorrelación residual, identificada mediante la prueba de Ljung-Box en los tres rezagos evaluados ($p < 0.001$), evidencia la presencia de dependencia temporal que no fue completamente capturada por el modelo. Este comportamiento podría estar relacionado con la exclusión operativa de los rezagos intradía recientes ($k < 32$) y con la superposición del horizonte de predicción ($h = 2$), factores que pueden contribuir a trasladar parte de las fluctuaciones de corto plazo al término de error. Por su parte, la detección de efectos ARCH significativos ($p < 0.001$) indica que la variabilidad residual conserva una estructura temporal, con períodos de mayor y menor dispersión que no son completamente explicados por las predicciones. Aunque la función de pérdida Tweedie incorpora una relación entre la media y la varianza condicional $\left(\operatorname{Var}(y\mid X) = \phi\,\mu^{1.4311}\right)$, esta especificación no garantiza la eliminación de la heterocedasticidad condicional ni de la dependencia temporal en los residuos.

La inspección visual evidenció que el modelo LightGBM-Tweedie reprodujo los ciclos estacionales regulares de la operación de SOBUSA, las mesetas de demanda en días hábiles y los valles de fin de semana. También capturó las contracciones atípicas registradas entre 2024 y 2025, incluyendo las caídas de cierre de año, Semana Santa y la interrupción operativa de septiembre de 2024.

In [14]:
ts.plot_temporal_fit_panel(
    dates=X_train_full.index[-window:],
    y_true=y_train_full[-window:],
    y_pred=preds_train_final,
    zoom_left=("2025-02-02", "2025-02-08"),
    zoom_right=("2025-02-28", "2025-02-28"),
    text_left="Febrero 2025",
    text_right="28 Febrero 2025",
    title="Demanda de pasajeros 2024-2025",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros por día",
    ylab_zoom="Pasajeros por franja",
    zoom_ylim_pad=(0.10, 0.12), top_ylim=(-2000, 75000),
    hspace=0.1, wspace = 0.05
)

A partir de febrero de 2025 se presnetó una subestimación de la demanda, particularmente durante las jornadas de mayor afluencia. Aunque el modelo reprodujo las oscilaciones de la serie diaria, presentó dificultades para alcanzar los máximos observados, con diferencias más pronunciadas durante algunos períodos de marzo y abril de 2025. Este desfase podría estar relacionado con la amplitud de la ventana de entrenamiento ($26000$ franjas), que incorpora los distintos regímenes de demanda registrados durante 2024, y con el suavizado hacia la media condicional propio del ensamble, que limita la reproducción de incrementos abruptos y tiende a moderar las predicciones durante los períodos de mayor actividad.

En el detalle intradía de la semana del 2 al 8 de febrero de 2025, la subestimación se concentró en los picos matutinos de los días hábiles, con predicciones cercanas a $800\text{--}1000$ pasajeros por franja frente a máximos reales aproximados de $1100\text{--}1300$. La discrepancia fue más evidente durante el 28 de febrero, cuando la demanda real superó los $1000$ pasajeros en las primeras horas de la mañana, mientras que el modelo apenas alcanzó valores cercanos a $350\text{--}400$, manteniéndose por debajo de la serie observada durante gran parte de la jornada. A pesar de estas diferencias, LightGBM-Tweedie capturó la periodicidad semanal, las transiciones entre los picos de demanda y el descenso nocturno, aunque con un suavizamiento considerable de las fluctuaciones intradía.

### Evaluación

La evaluación del rendimiento del modelo LightGBM-Tweedie contrasta su capacidad de ajuste sobre los datos observados con su capacidad predictiva sobre demanda no vista. Mientras que la evaluación dentro de la muestra (*In-Sample*, sobre la ventana activa $w = 26{,}000$ franjas del cierre de entrenamiento) verifica la convergencia del algoritmo mediante un ajuste estático, la evaluación fuera de la muestra (*Out-of-Sample*, conjunto *holdout* del `2025-05-26` al `2025-12-31`, $N = 14{,}247$ franjas) determina la utilidad operativa real ante nuevos regímenes de movilidad mediante reentrenamiento secuencial.

Para diagnosticar la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*), se aplicaron los criterios establecidos en el protocolo experimental. Estos criterios se basan en el error relativo porcentual entre las métricas de prueba y entrenamiento, denotado como:

$$
\Delta_{\text{rel}} (\%) = \frac{M_{\text{test}} - M_{\text{train}}}{|M_{\text{train}}|} \times 100
$$

Se estableció un umbral de tolerancia relativo de $|\Delta_{\text{rel}}| = 5\%$ sobre la función de pérdida principal (*Tweedie Deviance*, con parámetro de potencia óptimo $p = 1.4311$). Un incremento relativo superior a este margen evidenciaría degradación en la capacidad de generalización.

En el contexto del pronóstico de demanda de pasajeros bajo verosimilitud Tweedie, debe considerarse la naturaleza asimétrica de esta función de pérdida y su sensibilidad diferenciada entre franjas de baja afluencia (incluyendo cierres operativos en cero) y horas pico. Dado que un desplazamiento puntual en la demanda real afecta de forma no lineal las métricas agregadas, la lectura de $\Delta_{\text{rel}}$ se complementó con métricas de escala (`MAE`, `RMSE`), métricas relativas (`SMAPE`, `MASE`), el diagnóstico de sesgo (`Bias`) y la inspección visual de las trayectorias diarias e intradía.

Las predicciones fuera de muestra se obtuvieron ejecutando el método `forecast` de `LightGBMTweedieWrapper` sobre la serie continua concatenada (garantizando la disponibilidad exacta de los rezagos operativos sin pérdida de observaciones iniciales ni fuga de información). Este procedimiento aplicó un esquema de reentrenamiento periódico por bloques (`n_splits = 5`) con ventana deslizante fija (`window = 26000`, `embargo = 1`), generando pronósticos puntuales en la escala original de pasajeros por franja de 15 minutos. A diferencia del ajuste estático dentro de la muestra, la actualización progresiva de la ventana deslizante incorpora el repunte del primer semestre de 2025.

In [15]:
df_eval_context = pd.concat([train, holdout], axis=0)
X_eval_all, y_eval_all, y_naive_eval_all, _ = trainer._build_dataset(df_eval_context)

holdout_mask = X_eval_all.index >= holdout.index.min()
X_holdout_final = X_eval_all.loc[holdout_mask]
y_holdout_final = y_eval_all[holdout_mask]
y_naive_holdout_final = y_naive_eval_all[holdout_mask]

start_time = time.time()
preds_holdout = model_production.forecast(
    X_seq=X_holdout_final,
    y_seq=y_holdout_final,
    window=window,
    embargo=trainer.embargo,
    n_splits=trainer.n_splits,
)
preds_time = time.time() - start_time

df_preds = pd.DataFrame({
    "date": X_holdout_final.index,
    "y_true": y_holdout_final,
    "y_naive": y_naive_holdout_final,
    "y_pred": preds_holdout,
})

output_path = RESULTS_PATH / "lgbm_holdout_predictions_30.csv"
df_preds.to_csv(output_path, index=False)

El análisis de los residuos fuera de muestra ($e_t = y_t - \hat{y}_t$, evaluado sobre las últimas $1000$ franjas del conjunto *holdout*) evidenció persistencia de estructura temporal y heterocedasticidad condicional en el error de pronóstico. Las pruebas de Ljung-Box rechazaron la hipótesis nula de ausencia de autocorrelación en todos los rezagos evaluados $\left(p_\text{adj} < .001\right)$, indicando dependencia serial remanente de corto y mediano plazo. De forma paralela, la prueba ARCH-LM detectó efectos ARCH significativos en los tres rezagos considerados $\left(p_\text{adj} < .001\right)$. Estos resultados indican que, aunque el modelo reproduce los principales patrones diarios e intradía de la demanda de la ruta K54, persisten fluctuaciones de corto plazo y agrupamientos de variabilidad residual que no son completamente capturados por el ensamble. Este comportamiento podría estar relacionado con las variaciones operativas durante las franjas de mayor afluencia y con la superposición del horizonte de predicción de 30 minutos ($h=2$), mientras que la especificación Tweedie $\left(\operatorname{Var}(y\mid X)=\phi\,\mu^{1.4311}\right)$ permite representar la relación entre la media y la dispersión condicional, sin garantizar la eliminación de la dependencia temporal en los errores.

In [16]:
residuals_test = (
    df_preds["y_true"].to_numpy() - df_preds["y_pred"].to_numpy()
)

block_size = 1000
residuals_block = residuals_test[-block_size:]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_test = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_test,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),1638.425,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),1958.238,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),2022.722,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),434.031,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),437.139,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),439.405,< .001,< .001,Efectos ARCH Significativos


De acuerdo con los criterios establecidos en el protocolo experimental, no se observa evidencia de sobreajuste (*overfitting*) severo ni de subajuste (*underfitting*) en el modelo LightGBM-Tweedie. La función de pérdida principal (*Tweedie Deviance*) presenta una reducción relativa del $4.79\%$ entre entrenamiento y prueba, pasando de $2.4217$ a $2.3058$, lo que respalda una capacidad de generalización razonable fuera de muestra. Este comportamiento contrasta con los incrementos moderados registrados en las métricas `MAE` ($7.24\%$), `RMSE` ($3.48\%$), `MSE` ($7.08\%$) y `Pinball` ($7.24\%$), mientras que el `SMAPE` disminuye ligeramente ($0.55\%$). 

Las mayores discrepancias se presentan en el `MASE`, que aumenta un $22.94\%$ (de $0.5404$ a $0.6644$), y en el `Bias`, que cambia de $-23.41$ a $24.92$ pasajeros, indicando una transición de subestimación promedio durante el entrenamiento a sobreestimación fuera de muestra. Esta inversión del sesgo sugiere diferencias en los patrones de demanda entre ambos períodos, posiblemente asociadas con los cambios de régimen identificados en el EDA de la ruta K54. Por su parte, el incremento del `MASE` indica una reducción de la ventaja relativa frente al pronóstico ingenuo estacional, aunque sus valores inferiores a uno confirman que el modelo mantiene un desempeño superior al referente utilizado en ambos conjuntos.

In [18]:
df_preds = pd.read_csv(RESULTS_PATH / "lgbm_holdout_predictions_30.csv")

X_train_full, y_train_full, y_naive_train_full, _ = trainer._build_dataset(train)
preds_train_final = model_production.predict_in_sample(X_train_full.iloc[-window:])

ts.display_train_test_comparison(
    y_train=y_train_full[-window:],
    preds_train=preds_train_final,
    y_naive_train=y_naive_train_full[-window:],
    y_test=df_preds["y_true"].to_numpy(),
    preds_test=df_preds["y_pred"].to_numpy(),
    y_naive_test=df_preds["y_naive"].to_numpy(),
    p=1.4311, error_relativo=True
)

Métrica,Entrenamiento (In-Sample),Prueba (Out-of-Sample),Error Relativo (%)
Tweedie Deviance,2.4217,2.3058,-4.7863%
MAE,90.0083,96.5230,+7.2379%
RMSE,132.7321,137.3520,+3.4806%
MSE,17617.8166,18865.5697,+7.0823%
Bias,-23.4148,24.9191,+206.4243%
SMAPE,0.1820,0.1810,-0.5452%
MASE,0.5404,0.6644,+22.9363%
Pinball (τ=0.50),45.0041,48.2615,+7.2379%


La inspección visual de las trayectorias predichas frente a la demanda realizada en el conjunto *holdout* muestra que el modelo LightGBM-Tweedie reproduce con alta fidelidad la envolvente diaria, la estacionalidad semanal y el perfil multimodal intradía a lo largo del segundo semestre. A escala agregada diaria (panel superior), el ajuste sigue con precisión los principales patrones de demanda operativa en días hábiles, aunque presenta episodios puntuales de sobreestimación y subestimación asociados a variaciones atípicas de la demanda, particularmente durante junio, julio, octubre y los cierres operativos de final de diciembre. En general, las predicciones mantienen una correspondencia estrecha con la serie observada, reproduciendo tanto los máximos de las jornadas laborales como las caídas recurrentes de menor actividad.

A escala de franjas de 15 minutos (paneles inferiores, correspondientes a la primera semana de septiembre y al 15 de octubre de 2025), se observa una ligera sobreestimación en algunos picos matutinos de septiembre, mientras que el 15 de octubre persiste una subestimación en la cúspide de la hora pico matutina (alrededor de las 05:00–05:30), con predicciones cercanas a $1500\text{--}1600$ pasajeros por franja frente a máximos reales aproximados de $1700\text{--}1750$, aunque de magnitud sustancialmente menor que la registrada durante el entrenamiento. Este comportamiento es consistente con el efecto de suavizado propio de la estimación de la media condicional mediante árboles de decisión frente a fluctuaciones de alta frecuencia, sin comprometer la captura de la meseta de media mañana ni el repunte vespertino.

In [19]:
ts.plot_temporal_fit_panel(
    dates=pd.to_datetime(df_preds["date"]),
    y_true=df_preds["y_true"].to_numpy(),
    y_pred=df_preds["y_pred"].to_numpy(),
    zoom_left=("2025-09-01", "2025-09-07"),
    zoom_right=("2025-10-15", "2025-10-15"),
    text_left="Septiembre 2025",
    text_right="15 Octubre 2025",
    title="Demanda de pasajeros Holdout 2025 (Out-of-Sample)",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros por día",
    ylab_zoom="Pasajeros por franja",
    zoom_ylim_pad=(0.10, 0.12),
    hspace=0.1,
    wspace=0.05,
)

## Ventana de predicción de 60 minutos

### Entrenamiento

Al aplicar el estimador por momentos sobre los conteos acumulados de 60 minutos ($h=4$) en la ventana inicial de entrenamiento, se obtuvo un parámetro de potencia de $p=1.632635186036709$. Este valor sitúa la distribución dentro de la familia Tweedie compuesta Poisson-Gamma ($1<p<2$), permitiendo representar la asimetría positiva y la relación entre la media y la varianza de la demanda. En la inspección visual, la curva teórica suaviza la estructura multimodal observada en los datos, particularmente las concentraciones asociadas a distintos niveles de actividad operativa. Por su parte, el Q-Q plot muestra una alineación consistente en los cuantiles centrales, con desviaciones en la cola superior que sugieren una menor capacidad de la distribución ajustada para reproducir los valores extremos de demanda.

In [22]:
warmup_data = train["pasajeros"].iloc[:26000]

pasajeros_h4_warmup = (
    warmup_data
    .rolling(window=4)
    .sum()
    .shift(-3)
    .dropna()
)

p_warmup_h4 = tt._estimate_tweedie_p_internal(pasajeros_h4_warmup)

_, _ = tt.plot_tweedie_fit(
    series=pasajeros_h4_warmup,
    p=p_warmup_h4,
    titles=("Distribución de Pasajeros (Warmup)", "Q-Q Plot (Warmup)"),
    xlab="Pasajeros acumulados en 60 min (h=4)",
    pads=(15, 15, 12),
    legend_bbox=(1.0, 0.95)
)

Para la selección de hiperparámetros del horizonte de 60 minutos ($h=4$ franjas), se replicó el protocolo de validación cruzada temporal de 5 pliegues con ventana deslizante (*sliding window*) y el mismo espacio de búsqueda en Optuna utilizado en el horizonte previo. A diferencia del caso anterior, el parámetro de potencia de la desviación Tweedie se fijó en $p=1.632635186036709$, estimado sobre el bloque de calentamiento para $h=4$, y el margen de embargo entre entrenamiento y validación se amplió a tres franjas ($h-1=3$), evitando el solapamiento de las ventanas acumuladas de una hora entre ambos conjuntos y reduciendo el riesgo de fuga de información.

In [23]:
exec_id = 60
exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=exec_dir,
    feature_builder=feature_builder,
    horizon_slots=4,
    n_splits=5,
    warmup=WINDOW_MAX,
    embargo=3,
    n_trials=100,
    suggest_params_fn=lgbm_search_space,
    early_stop_rounds=30,
    inner_val_frac=0.1,
    model_kwargs={
        "tweedie_variance_power": round(float(p_warmup), 4),
        "n_estimators": 5000,
        "early_stopping_rounds": 300,
        "num_threads": 8,
        "seed": 42,
    },
)

study = trainer.run(train)
best_params = dict(study.best_params)

  0%|          | 0/100 [00:00<?, ?it/s]

El proceso de optimización ha alcanzado el ensayo 85 de los 100 programados, sin que hasta el momento se haya activado la parada temprana. La mejor configuración se identificó en el ensayo 54, alcanzando una Desviación Tweedie mínima de 4.9786. Durante la exploración se descartaron 5 configuraciones mediante el mecanismo de poda (*pruning*), mientras que las 80 restantes finalizaron satisfactoriamente y ninguna presentó fallos durante su ejecución.

El ajuste de los árboles a través de los 400 pliegues ejecutados (80 ensayos completados $\times$ 5 particiones) demandó un tiempo neto de 28.41 minutos (1,704.67 segundos), equivalente a un promedio de 4.13 segundos por pliegue. La ejecución global acumula 33 minutos y 18 segundos (1,998 segundos); el margen restante de aproximadamente 4.89 minutos (293.33 segundos) corresponde al tiempo adicional del procedimiento, que puede incluir el entrenamiento inicial sobre el bloque de calentamiento, el cómputo de las métricas de validación en memoria y la gestión de los ensayos.

In [24]:
resumen_data = []

exec_id = 60
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_tweedie = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(
            study_name="LightGBMTweedieWrapper_h4_study",
            storage=storage_url
        )

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])
        
        mejor_tweedie = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Fallidos": fallidos,
    "Mejor Tweedie Dev": mejor_tweedie,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Fallidos,Mejor Tweedie Dev,Tiempo Total (s),Tiempo por Fold (s)
85,80,5,0,4.9786,1704.6700,4.1280


El ranking de configuraciones mostró que las cinco mejores opciones presentaron una Desviación Tweedie entre 4.979 y 4.992. Dado que la variabilidad entre los pliegues de validación se mantuvo elevada ($\sigma \approx 1.70\text{--}1.75$), las diferencias de centésimas entre estas alternativas resultan reducidas frente a la dispersión observada, aunque no permiten afirmar un empate estadístico sin una comparación formal. Asimismo, la exploración exhibió una convergencia estructural hacia la ventana máxima evaluada de 26000 franjas, árboles de elevada capacidad expresiva (entre 130 y 239 hojas), un submuestreo estricto de predictores (`feature_fraction` fijado de forma unánime en 0.40), fracciones bajas de muestreo de filas (`bagging_fraction` entre 0.40 y 0.45), tasas de aprendizaje muy reducidas (entre 0.0014 y 0.0025) y penalizaciones $\ell_1$ y $\ell_2$ cercanas a cero.

La elección consistente de 26000 franjas sugiere la conveniencia de abarcar más de un año calendario para capturar la estacionalidad anual y disponer de suficiente historial sobre los patrones semanales sin degradar excesivamente la adaptación a los regímenes recientes. A su vez, el acoplamiento de tasas de aprendizaje reducidas con el submuestreo pronunciado de variables y observaciones permitió regularizar el ensamble pese a la mayor complejidad de los árboles, reduciendo la necesidad de imponer penalizaciones explícitas elevadas sobre los pesos de las hojas.

In [30]:
exec_dir = RESULTS_PATH / "exec_60"
df_lgbm = pd.read_csv(exec_dir / "LightGBMTweedieWrapper_results.csv")

df_lgbm["lr"] = df_lgbm["learning_rate"].apply(lambda x: f"{x:.4f}")
df_lgbm["l1"] = df_lgbm["lambda_l1"].apply(lambda x: f"{x:.1e}")
df_lgbm["l2"] = df_lgbm["lambda_l2"].apply(lambda x: f"{x:.1e}")

cols_agrupar = [
    "window", 
    "num_leaves", 
    "min_data_in_leaf", 
    "feature_fraction", 
    "bagging_fraction", 
    "lr", 
    "l1",
    "l2"
]

df_agg = df_lgbm.groupby(cols_agrupar)["tweedie_deviance"].agg(["mean", "std"]).reset_index()
df_agg["std"] = df_agg["std"].fillna(0.0)

df_agg["Tweedie Deviance"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)

df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "window": "Ventana",
    "num_leaves": "Hojas",
    "min_data_in_leaf": "Min Data Hoja",
    "feature_fraction": "Feat. Frac",
    "bagging_fraction": "Bagging Frac",
    "lr": "LR",
    "l1": "L1",
    "l2": "L2"
})

df_final = df_top5[
    ["Ventana", "Hojas", "Min Data Hoja", "Feat. Frac", "Bagging Frac", "LR", "L1", "L2", "Tweedie Deviance"]
]

tt.display_styled_df(df=df_final, decimals=4, align_left_first=False)

Ventana,Hojas,Min Data Hoja,Feat. Frac,Bagging Frac,LR,L1,L2,Tweedie Deviance
"26,000",159,69,0.4000,0.4000,0.0014,4.5e-04,1.5e-06,4.979 ± 1.719
"26,000",130,93,0.4000,0.4500,0.0023,1.8e-04,4.8e-07,4.985 ± 1.717
"26,000",177,87,0.4000,0.4000,0.0018,4.6e-04,6.1e-07,4.987 ± 1.751
"26,000",148,62,0.4000,0.4000,0.0025,1.8e-04,1.6e-07,4.988 ± 1.705
"26,000",239,102,0.4000,0.4500,0.0015,2.5e-03,8.0e-07,4.992 ± 1.720


La selección del modelo ganador entre las cinco mejores configuraciones se resolvió a favor de la primera (Desviación Tweedie de 4.979). Con una variabilidad entre pliegues ligeramente superior a la de la segunda alternativa ($\sigma = 1.719$ frente a $\sigma = 1.717$), el modelo seleccionado ofrece una mayor capacidad expresiva mediante árboles de 159 hojas frente a las 130 de su competidor más cercano, aunque mantiene un control sobre su complejidad al exigir un mínimo de 69 observaciones por hoja. Ambas configuraciones conservan el submuestreo estricto de predictores (`feature_fraction` = 0.40), mientras que el modelo ganador emplea una fracción de muestreo de observaciones ligeramente inferior (`bagging_fraction` = 0.40 frente a 0.45) y una tasa de aprendizaje más reducida (0.0014 frente a 0.0023), favoreciendo un ajuste gradual del ensamble. Asimismo, las penalizaciones $\ell_1$ y $\ell_2$ permanecen próximas a cero, por lo que la regularización depende principalmente del submuestreo de variables y observaciones, junto con las restricciones estructurales de los árboles.

In [2]:
exec_dir = RESULTS_PATH / "exec_60"
storage_url = f"sqlite:///{(exec_dir / 'optuna_study.db').resolve()}"
study = optuna.load_study(
    study_name="LightGBMTweedieWrapper_h4_study",
    storage=storage_url,
)
best_params = dict(study.best_params)

window = best_params["window"]
model_params = {k: v for k, v in best_params.items() if k != "window"}

final_kwargs = {
    "tweedie_variance_power": 1.632635186036709,
    "n_estimators": 5000,
    "early_stopping_rounds": 300,
    "inner_val_frac": 0.3,
    "num_threads": 8,
    "seed": 42,
}

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=exec_dir,
    feature_builder=feature_builder,
    horizon_slots=4,
    n_splits=5,
    warmup=26000,
    embargo=3,
    n_trials=100,
    model_kwargs={},
)

X_train_full, y_train_full, _, _ = trainer._build_dataset(train)

model_production = LightGBMTweedieWrapper(**model_params, **final_kwargs)
model_production.fit(X_train_full.iloc[-window:], y_train_full[-window:])

Al evaluar los residuales estandarizados no solapados del bloque final de entrenamiento, las pruebas de diagnóstico evidencian la persistencia de dependencia temporal en los errores de pronóstico, aunque sin indicios estadísticamente significativos de heterocedasticidad condicional. Durante los primeros diez rezagos se detecta autocorrelación significativa tras aplicar la corrección de Holm ($p_{\text{adj}}=0.024$), mientras que la prueba ARCH-LM no identifica agrupamiento de volatilidad significativo ($p_{\text{adj}}=0.593$).

En los rezagos 20 y 30 persiste la autocorrelación significativa ($p_{\text{adj}}=0.024$), con estadísticos de Ljung-Box que aumentan de 38.307 a 50.365, indicando que el modelo no captura completamente la estructura serial de los errores. Por el contrario, las pruebas ARCH-LM mantienen valores ajustados superiores al nivel de significancia ($p_{\text{adj}}=0.473$), por lo que no existe evidencia suficiente para rechazar la ausencia de efectos ARCH. Estos resultados sugieren que, aunque LightGBM-Tweedie logra estabilizar la variabilidad residual, todavía persisten patrones de dependencia temporal que podrían estar relacionados con la dinámica operativa de la ruta El Silencio.

In [31]:
preds_train_final = model_production.predict_in_sample(X_train_full.iloc[-window:])
residuals_train = y_train_full[-window:] - preds_train_final

block_size = 1000
horizon_slots = 4
residuals_block = residuals_train[-block_size::horizon_slots]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_train = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_train,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),22.069,.015,.024,Autocorrelación Significativa
Ljung-Box (Lag 20),38.307,.008,.024,Autocorrelación Significativa
Ljung-Box (Lag 30),50.365,.011,.024,Autocorrelación Significativa
ARCH-LM (Lag 10),8.364,.593,.593,Sin Efectos ARCH
ARCH-LM (Lag 20),26.249,.158,.473,Sin Efectos ARCH
ARCH-LM (Lag 30),36.884,.181,.473,Sin Efectos ARCH


En la inspección visual de la demanda predicha en la muestra de entrenamiento, el modelo reproduce con fidelidad el ciclo semanal y los cambios estacionales de nivel a lo largo de 2024, capturando tanto las caídas de fines de semana como los repuntes de agosto a noviembre. Al igual que en el horizonte de 30 minutos, se observa una tendencia sistemática a subestimar la demanda durante los primeros meses de 2025, donde la serie real alcanza picos superiores a los 110000 pasajeros acumulados mientras las predicciones se estabilizan alrededor de 95000.

Esta subestimación se detalla en los paneles inferiores correspondientes a febrero de 2025. En la ventana semanal del 2 al 8 de febrero, el ensamble logra replicar la forma de las ondas diarias y la desaceleración del sábado, pero atenúa la amplitud de los picos matutinos, con máximos predichos cercanos a 2500 pasajeros frente a valores reales superiores a 3000. En el perfil intradía del 28 de febrero, las estimaciones reproducen parcialmente la trayectoria de la hora pico temprana y la meseta del mediodía, aunque permanecen considerablemente por debajo de la curva observada, convergiendo hacia los valores reales únicamente en las horas de cierre vespertino.

In [37]:
ts.plot_temporal_fit_panel(
    dates=X_train_full.index[-window:],
    y_true=y_train_full[-window:],
    y_pred=preds_train_final,
    zoom_left=("2025-02-02", "2025-02-08"),
    zoom_right=("2025-02-28", "2025-02-28"),
    text_left="Febrero 2025",
    text_right="28 Febrero 2025",
    title="Demanda acumulada de pasajeros 2024-2025 (60 min)",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros acumulados",
    ylab_zoom="Pasajeros por ventana (60 min)",
    zoom_ylim_pad=(0.10, 0.12),
    top_ylim= (-2000, 145000), 
    hspace=0.1,
    wspace=0.05,
)

### Evaluación

Para el horizonte de 60 minutos ($h = 4$), se aplicó el mismo protocolo de evaluación comparativa y criterios de diagnóstico establecidos en la ventana previa, ajustando el parámetro de potencia de la función de pérdida Tweedie a $p = 1.632635186036709$. Las predicciones fuera de muestra sobre el conjunto *holdout* se generaron mediante el esquema de reentrenamiento secuencial por bloques (`n_splits = 5`, `window = 26000`), ampliando el margen de *embargo* a tres franjas (`embargo = 3`) para evitar el solapamiento entre las ventanas acumuladas de una hora. De este modo, los pronósticos puntuales se obtuvieron directamente en la escala de pasajeros acumulados por bloque de 60 minutos, incorporando de forma progresiva el cambio de nivel registrado durante el primer semestre de 2025.

In [38]:
df_eval_context = pd.concat([train, holdout], axis=0)
X_eval_all, y_eval_all, y_naive_eval_all, _ = trainer._build_dataset(df_eval_context)

holdout_mask = X_eval_all.index >= holdout.index.min()
X_holdout_final = X_eval_all.loc[holdout_mask]
y_holdout_final = y_eval_all[holdout_mask]
y_naive_holdout_final = y_naive_eval_all[holdout_mask]

start_time = time.time()
preds_holdout = model_production.forecast(
    X_seq=X_holdout_final,
    y_seq=y_holdout_final,
    window=window,
    embargo=trainer.embargo,
    n_splits=trainer.n_splits,
)
preds_time = time.time() - start_time

df_preds = pd.DataFrame({
    "date": X_holdout_final.index,
    "y_true": y_holdout_final,
    "y_naive": y_naive_holdout_final,
    "y_pred": preds_holdout,
})

output_path = RESULTS_PATH / "lgbm_holdout_predictions_60.csv"
df_preds.to_csv(output_path, index=False)

En el diagnóstico de los residuales estandarizados no solapados sobre el conjunto *holdout*, tanto las pruebas de Ljung-Box como las pruebas ARCH-LM resultan estadísticamente significativas en todos los rezagos evaluados tras aplicar la corrección de Holm ($p_{\text{adj}}<.001$). A diferencia del cierre de entrenamiento, donde se identificó dependencia serial significativa pero no efectos ARCH, el desempeño fuera de muestra exhibe tanto autocorrelación persistente como agrupamiento de volatilidad desde los primeros rezagos evaluados.

Esta dinámica residual podría estar relacionada con los desplazamientos de nivel y las perturbaciones operativas no observadas durante el segundo semestre de 2025, capaces de generar rachas consecutivas de subestimación o sobreestimación entre bloques de reentrenamiento. Debido a que el ensamble estima la media condicional sin modelar explícitamente la estructura autorregresiva ni la heterocedasticidad condicional del error, los resultados sugieren que persisten patrones de dependencia temporal y variabilidad residual no capturados completamente por el modelo en escenarios de demanda no estacionaria.

In [39]:
residuals_test = (
    df_preds["y_true"].to_numpy() - df_preds["y_pred"].to_numpy()
)

block_size = 1000
horizon_slots = 4
residuals_block = residuals_test[-block_size::horizon_slots]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_test = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_test,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),209.186,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),254.276,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),267.348,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),134.698,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),137.346,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),134.643,< .001,< .001,Efectos ARCH Significativos


De acuerdo con los criterios establecidos en el protocolo experimental, no se observa evidencia de sobreajuste (*overfitting*) severo ni de subajuste (*underfitting*) en el modelo LightGBM-Tweedie. La función de pérdida principal (*Tweedie Deviance*) presenta una reducción relativa del $20.90\%$ entre entrenamiento y prueba, pasando de $0.7803$ a $0.6173$, lo que respalda una capacidad de generalización favorable fuera de muestra. Este comportamiento es consistente con las reducciones registradas en las métricas `MAE` ($38.79\%$), `RMSE` ($44.56\%$), `MSE` ($69.26\%$) y `Pinball` ($38.79\%$), aunque el `SMAPE` aumenta un $19.61\%$, evidenciando una mejora en los errores absolutos y cuadráticos que no se extiende a la precisión relativa del pronóstico.

Las principales discrepancias se presentan en el `MASE`, que aumenta un $24.00\%$ (de $0.5293$ a $0.6563$), y en el `Bias`, que cambia de $-48.63$ a $40.12$ pasajeros, indicando una transición de subestimación promedio durante el entrenamiento a sobreestimación fuera de muestra. Esta inversión del sesgo sugiere diferencias en los patrones de demanda entre ambos períodos, posiblemente asociadas con los cambios de régimen identificados en el EDA de la ruta El Silencio. Por su parte, el incremento del `MASE` indica una reducción de la ventaja relativa frente al pronóstico ingenuo estacional, aunque sus valores inferiores a uno confirman que el modelo conserva un desempeño superior al referente utilizado en ambos conjuntos.

In [21]:
df_preds = pd.read_csv(RESULTS_PATH / "lgbm_holdout_predictions_60.csv")

X_train_full, y_train_full, y_naive_train_full, _ = trainer._build_dataset(train)
preds_train_final = model_production.predict_in_sample(X_train_full.iloc[-window:])

ts.display_train_test_comparison(
    y_train=y_train_full[-window:],
    preds_train=preds_train_final,
    y_naive_train=y_naive_train_full[-window:],
    y_test=df_preds["y_true"].to_numpy(),
    preds_test=df_preds["y_pred"].to_numpy(),
    y_naive_test=df_preds["y_naive"].to_numpy(),
    p=1.632635186036709,
    error_relativo=True,
)

Métrica,Entrenamiento (In-Sample),Prueba (Out-of-Sample),Error Relativo (%)
Tweedie Deviance,0.7803,0.6173,-20.8977%
MAE,245.6584,150.3688,-38.7895%
RMSE,398.4127,220.8848,-44.5588%
MSE,158732.6632,48790.0741,-69.2627%
Bias,-48.6343,40.1152,+182.4834%
SMAPE,0.1143,0.1367,+19.6073%
MASE,0.5293,0.6563,+24.0001%
Pinball (τ=0.50),122.8292,75.1844,-38.7895%


La inspección visual de las trayectorias predichas frente a la demanda realizada en el conjunto *holdout* muestra que el modelo reproduce con alta fidelidad la envolvente diaria, la estacionalidad semanal y el perfil multimodal intradía a lo largo del segundo semestre. A escala agregada diaria (panel superior), el ajuste sigue con precisión los principales patrones de demanda operativa en días hábiles, aunque presenta episodios puntuales de sobreestimación y subestimación asociados a variaciones atípicas de la demanda, particularmente durante junio, julio, octubre y los cierres operativos de final de diciembre. En general, las predicciones mantienen una correspondencia estrecha con la serie observada, reproduciendo tanto los máximos de las jornadas laborales como las caídas recurrentes de menor actividad.

A escala de ventanas acumuladas de 60 minutos (paneles inferiores, correspondientes a la primera semana de septiembre y al 15 de octubre de 2025), se observa una ligera sobreestimación en algunos picos matutinos de septiembre, mientras que el 15 de octubre las predicciones reproducen con precisión la cúspide de la hora pico matutina (alrededor de las 05:00–05:30), alcanzando valores cercanos a $2900\text{--}3000$ pasajeros frente a máximos reales aproximados de $3000\text{--}3100$. Este comportamiento evidencia una adecuada captura de los principales patrones intradía, con discrepancias puntuales durante la meseta de media mañana y el repunte vespertino, asociadas al suavizado de las fluctuaciones de alta frecuencia propio de la estimación de la media condicional mediante árboles de decisión.

In [23]:
ts.plot_temporal_fit_panel(
    dates=pd.to_datetime(df_preds["date"]),
    y_true=df_preds["y_true"].to_numpy(),
    y_pred=df_preds["y_pred"].to_numpy(),
    zoom_left=("2025-09-01", "2025-09-07"),
    zoom_right=("2025-10-15", "2025-10-15"),
    text_left="Septiembre 2025",
    text_right="15 Octubre 2025",
    title="Demanda acumulada de pasajeros Holdout 2025 (60 min)",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros acumulados",
    ylab_zoom="Pasajeros por ventana (60 min)",
    zoom_ylim_pad=(0.10, 0.12),
    hspace=0.1,
    wspace=0.05,
)

## Ventana de predicción de 120 minutos

### Entrenamiento

Al aplicar el estimador por momentos sobre los conteos acumulados de 120 minutos ($h=8$) en la ventana inicial de entrenamiento, se obtuvo un parámetro de potencia de $p$ pendiente de determinar. Este valor permitirá caracterizar la relación entre la media y la varianza de la demanda y, si se encuentra dentro del intervalo $1<p<2$, situará la distribución en la familia Tweedie compuesta Poisson-Gamma. En la inspección visual se evaluará la capacidad de la curva teórica para representar la estructura multimodal observada en los datos, particularmente las concentraciones asociadas a distintos niveles de actividad operativa. Por su parte, el Q-Q plot permitirá examinar la alineación de los cuantiles centrales y las posibles desviaciones en la cola superior, asociadas a la representación de los valores extremos de demanda.

In [4]:
warmup_data = train["pasajeros"].iloc[:26000]
pasajeros_h8_warmup = warmup_data.rolling(window=8).sum().shift(-7).dropna()

p_warmup_h8 = tt._estimate_tweedie_p_internal(pasajeros_h8_warmup)

_, _ = tt.plot_tweedie_fit(
    series=pasajeros_h8_warmup,
    p=p_warmup_h8,
    titles=("Distribución de Pasajeros (Warmup)", "Q-Q Plot (Warmup)"),
    xlab="Pasajeros acumulados en 120 min (h=8)",
    pads=(15, 15, 12),
    legend_bbox=(1.0, 0.95)
)


Para la selección de hiperparámetros del horizonte de 120 minutos ($h=8$ franjas), se replicó el protocolo de validación cruzada temporal de 5 pliegues con ventana deslizante (*sliding window*) y el mismo espacio de búsqueda en Optuna utilizado en los horizontes previos. A diferencia del caso anterior, el parámetro de potencia de la desviación Tweedie se fijó en $p=1.950592166020216$, estimado sobre el bloque de calentamiento para $h=8$, y el margen de embargo entre entrenamiento y validación se amplió a siete franjas ($h-1=7$), evitando el solapamiento de las ventanas acumuladas de dos horas entre ambos conjuntos y reduciendo el riesgo de fuga de información.

In [5]:
exec_id = 120
exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=exec_dir,
    feature_builder=feature_builder,
    horizon_slots=8,
    n_splits=5,
    warmup=WINDOW_MAX,
    embargo=7,
    n_trials=100,
    suggest_params_fn=lgbm_search_space,
    early_stop_rounds=30,
    inner_val_frac=0.1,
    model_kwargs={
        "tweedie_variance_power": 1.950592166020216,
        "n_estimators": 5000,
        "early_stopping_rounds": 300,
        "num_threads": 8,
        "seed": 42,
    },
)

study = trainer.run(train)
best_params = dict(study.best_params)

  0%|          | 0/100 [00:00<?, ?it/s]

El proceso de optimización completó la totalidad de los 100 ensayos programados, sin que se activara el mecanismo de parada temprana. La mejor configuración se identificó en el ensayo 78, alcanzando una Desviación Tweedie mínima de 0.2422. Durante la exploración no se produjeron podas ni fallos, completándose satisfactoriamente las 100 configuraciones evaluadas sin encontrar una alternativa que superara el óptimo registrado.

El ajuste de los árboles a través de los 500 pliegues ejecutados (100 ensayos completados $\times$ 5 particiones) demandó un tiempo neto de 21.13 minutos (1,268.01 segundos), equivalente a un promedio de 2.54 segundos por pliegue. La ejecución global tomó 24 minutos y 32 segundos (1,472 segundos); el margen restante de aproximadamente 3.40 minutos (203.99 segundos) corresponde al tiempo adicional del procedimiento, que puede incluir el entrenamiento inicial sobre el bloque de calentamiento, el cómputo de las métricas de validación en memoria y la gestión de los ensayos.

In [6]:
resumen_data = []

exec_id = 120
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
mejor_tweedie = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)
else:
    print(f"No se encontró el CSV en {exec_dir}")

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(
            study_name="LightGBMTweedieWrapper_h8_study",
            storage=storage_url
        )

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])
        
        mejor_tweedie = round(study.best_value, 4)

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")
else:
    print(f"No se encontró la BD en {db_path}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Fallidos": fallidos,
    "Mejor Tweedie Dev": mejor_tweedie,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,Fallidos,Mejor Tweedie Dev,Tiempo Total (s),Tiempo por Fold (s)
100,100,0,0,0.2422,1268.0100,2.5360


El ranking de configuraciones mostró que las cinco mejores opciones presentaron una Desviación Tweedie en torno a 0.242. Dado que la variabilidad entre los pliegues de validación se mantuvo prácticamente constante ($\sigma \approx 0.258\text{--}0.259$), las diferencias entre estas alternativas resultan mínimas frente a la dispersión observada, aunque no permiten establecer un empate estadístico sin una comparación formal. Asimismo, la exploración exhibió una convergencia estructural hacia una ventana de 24000 franjas, árboles de complejidad moderada (entre 80 y 119 hojas), un submuestreo estricto de predictores (`feature_fraction` fijado de forma unánime en 0.40), fracciones altas de muestreo de filas (`bagging_fraction` entre 0.85 y 0.95), tasas de aprendizaje muy reducidas (entre 0.0020 y 0.0042) y penalizaciones $\ell_1$ bajas, acompañadas de penalizaciones $\ell_2$ de mayor magnitud.

La elección consistente de 24000 franjas sugiere la conveniencia de utilizar un historial amplio para capturar los patrones estacionales y semanales de la demanda, manteniendo cierta capacidad de adaptación a los regímenes recientes. A su vez, el acoplamiento de tasas de aprendizaje reducidas con el submuestreo pronunciado de variables, fracciones elevadas de observaciones y restricciones sobre el número mínimo de datos por hoja contribuye a controlar la complejidad del ensamble. Las penalizaciones $\ell_2$, comprendidas entre 0.082 y 0.74, proporcionan un mecanismo adicional de regularización sobre los pesos de las hojas.

In [7]:
exec_dir = RESULTS_PATH / "exec_120"
df_lgbm = pd.read_csv(exec_dir / "LightGBMTweedieWrapper_results.csv")

df_lgbm["lr"] = df_lgbm["learning_rate"].apply(lambda x: f"{x:.4f}")
df_lgbm["l1"] = df_lgbm["lambda_l1"].apply(lambda x: f"{x:.1e}")
df_lgbm["l2"] = df_lgbm["lambda_l2"].apply(lambda x: f"{x:.1e}")

cols_agrupar = [
    "window", 
    "num_leaves", 
    "min_data_in_leaf", 
    "feature_fraction", 
    "bagging_fraction", 
    "lr", 
    "l1",
    "l2"
]

df_agg = df_lgbm.groupby(cols_agrupar)["tweedie_deviance"].agg(["mean", "std"]).reset_index()
df_agg["std"] = df_agg["std"].fillna(0.0)

df_agg["Tweedie Deviance"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)

df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "window": "Ventana",
    "num_leaves": "Hojas",
    "min_data_in_leaf": "Min Data Hoja",
    "feature_fraction": "Feat. Frac",
    "bagging_fraction": "Bagging Frac",
    "lr": "LR",
    "l1": "L1",
    "l2": "L2"
})

df_final = df_top5[
    ["Ventana", "Hojas", "Min Data Hoja", "Feat. Frac", "Bagging Frac", "LR", "L1", "L2", "Tweedie Deviance"]
]

tt.display_styled_df(df=df_final, decimals=4, align_left_first=False)

Ventana,Hojas,Min Data Hoja,Feat. Frac,Bagging Frac,LR,L1,L2,Tweedie Deviance
"24,000",100,173,0.4000,0.9000,0.0025,5.7e-04,1.3e-01,0.242 ± 0.259
"24,000",80,197,0.4000,0.9500,0.0042,1.9e-03,2.3e-01,0.242 ± 0.259
"24,000",119,199,0.4000,0.9000,0.0035,1.3e-03,7.1e-01,0.242 ± 0.259
"24,000",89,157,0.4000,0.8500,0.0020,6.9e-03,7.4e-01,0.242 ± 0.258
"24,000",104,279,0.4000,0.9000,0.0034,1.8e-03,8.2e-02,0.242 ± 0.259


La selección del modelo ganador entre las cinco mejores configuraciones se resolvió a favor de la primera (Desviación Tweedie de 0.242). Con una variabilidad entre pliegues prácticamente idéntica a la de la segunda alternativa ($\sigma = 0.259$ en ambos casos), el modelo seleccionado presenta una mayor capacidad expresiva mediante árboles de 100 hojas frente a las 80 de su competidor más cercano, aunque mantiene un control sobre su complejidad al exigir un mínimo de 173 observaciones por hoja. Ambas configuraciones conservan el submuestreo estricto de predictores (`feature_fraction` = 0.40), mientras que el modelo ganador emplea una fracción de muestreo de observaciones ligeramente inferior (`bagging_fraction` = 0.90 frente a 0.95) y una tasa de aprendizaje más reducida (0.0025 frente a 0.0042), favoreciendo un ajuste gradual del ensamble. Asimismo, las penalizaciones $\ell_1$ permanecen próximas a cero, mientras que las penalizaciones $\ell_2$ presentan valores moderados (0.13 frente a 0.23), contribuyendo a regularizar los pesos de las hojas junto con el submuestreo de variables y observaciones y las restricciones estructurales de los árboles.

In [5]:
exec_dir = RESULTS_PATH / "exec_120"
storage_url = f"sqlite:///{(exec_dir / 'optuna_study.db').resolve()}"

study = optuna.load_study(
    study_name="LightGBMTweedieWrapper_h8_study",
    storage=storage_url,
)

best_params = dict(study.best_params)

window = best_params["window"]
model_params = {k: v for k, v in best_params.items() if k != "window"}

final_kwargs = {
    "tweedie_variance_power": 1.950592166020216,
    "n_estimators": 5000,
    "early_stopping_rounds": 300,
    "inner_val_frac": 0.3,
    "num_threads": 8,
    "seed": 42,
}

trainer = WalkForwardTrainer(
    model_class=LightGBMTweedieWrapper,
    scorer=DemandScorer(),
    output_dir=exec_dir,
    feature_builder=feature_builder,
    horizon_slots=8,
    n_splits=5,
    warmup=26000,
    embargo=7,
    n_trials=100,
    model_kwargs={},
)

X_train_full, y_train_full, _, _ = trainer._build_dataset(train)

model_production = LightGBMTweedieWrapper(**model_params, **final_kwargs)
model_production.fit(X_train_full.iloc[-window:], y_train_full[-window:])

Al evaluar los residuales estandarizados no solapados del bloque final de entrenamiento, las pruebas de diagnóstico evidencian la persistencia de dependencia temporal en los errores de pronóstico, aunque sin indicios estadísticamente significativos de heterocedasticidad condicional. Durante los primeros diez rezagos se detecta autocorrelación significativa tras aplicar la corrección de Holm ($p_{\text{adj}}=0.021$), mientras que la prueba ARCH-LM no identifica agrupamiento de volatilidad significativo ($p_{\text{adj}}=0.348$).

En los rezagos 20 y 30 persiste la autocorrelación significativa ($p_{\text{adj}}=0.021$), con estadísticos de Ljung-Box que aumentan de 37.925 a 52.302, indicando que el modelo no captura completamente la estructura serial de los errores. Por el contrario, las pruebas ARCH-LM mantienen valores ajustados superiores al nivel de significancia ($p_{\text{adj}}=0.833$), por lo que no existe evidencia suficiente para rechazar la ausencia de efectos ARCH. Estos resultados sugieren que, aunque LightGBM-Tweedie no presenta evidencia de agrupamiento significativo de la varianza residual, todavía persisten patrones de dependencia temporal que podrían estar relacionados con la dinámica operativa de la ruta El Silencio.

In [9]:
preds_train_final = model_production.predict_in_sample(X_train_full.iloc[-window:])
residuals_train = y_train_full[-window:] - preds_train_final

block_size = 1000
horizon_slots = 8
residuals_block = residuals_train[-block_size::horizon_slots]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_train = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_train,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),23.413,.009,.021,Autocorrelación Significativa
Ljung-Box (Lag 20),37.925,.009,.021,Autocorrelación Significativa
Ljung-Box (Lag 30),52.302,.007,.021,Autocorrelación Significativa
ARCH-LM (Lag 10),15.470,.116,.348,Sin Efectos ARCH
ARCH-LM (Lag 20),20.672,.417,.833,Sin Efectos ARCH
ARCH-LM (Lag 30),24.765,.736,.833,Sin Efectos ARCH


En la inspección visual de la demanda predicha en la muestra de entrenamiento, el modelo reproduce con fidelidad el ciclo semanal y los cambios estacionales de nivel a lo largo de 2024, capturando tanto las caídas de fines de semana como los repuntes de agosto a noviembre. Al igual que en los horizontes de 30 y 60 minutos, se observa una tendencia sistemática a subestimar la demanda durante los primeros meses de 2025, donde la serie real alcanza picos superiores a los 230000 pasajeros acumulados mientras las predicciones se estabilizan alrededor de 210000.

Esta subestimación se detalla en los paneles inferiores correspondientes a febrero de 2025. En la ventana semanal del 2 al 8 de febrero, el ensamble logra replicar la forma de las ondas diarias y la desaceleración del sábado, pero atenúa la amplitud de los picos matutinos, con máximos predichos cercanos a 5000 pasajeros frente a valores reales superiores a 5500. En el perfil intradía del 28 de febrero, las estimaciones reproducen parcialmente la trayectoria de la hora pico temprana y la meseta del mediodía, aunque permanecen considerablemente por debajo de la curva observada, convergiendo hacia los valores reales únicamente en las horas de cierre vespertino.

In [17]:
ts.plot_temporal_fit_panel(
    dates=X_train_full.index[-window:],
    y_true=y_train_full[-window:],
    y_pred=preds_train_final,
    zoom_left=("2025-02-02", "2025-02-08"),
    zoom_right=("2025-02-28", "2025-02-28"),
    text_left="Febrero 2025",
    text_right="28 Febrero 2025",
    title="Demanda acumulada de pasajeros 2024-2025 (60 min)",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros acumulados",
    ylab_zoom="Pasajeros por ventana (60 min)",
    zoom_ylim_pad=(0.10, 0.12),
    top_ylim= (-2000, 280000), 
    hspace=0.1,
    wspace=0.05,
)

### Evaluación

Para el horizonte de 120 minutos ($h = 8$), se aplicó el mismo protocolo de evaluación comparativa y criterios de diagnóstico establecidos en las ventanas previas, ajustando el parámetro de potencia de la función de pérdida Tweedie a $p = 1.950592166020216$. Las predicciones fuera de muestra sobre el conjunto *holdout* se generaron mediante el esquema de reentrenamiento secuencial por bloques (`n_splits = 5`, `window = 24000`), ampliando el margen de *embargo* a siete franjas (`embargo = 7`) para evitar el solapamiento entre las ventanas acumuladas de dos horas. De este modo, los pronósticos puntuales se obtuvieron directamente en la escala de pasajeros acumulados por bloque de 120 minutos, incorporando de forma progresiva el cambio de nivel registrado durante el primer semestre de 2025.

In [19]:
df_eval_context = pd.concat([train, holdout], axis=0)
X_eval_all, y_eval_all, y_naive_eval_all, _ = trainer._build_dataset(df_eval_context)

holdout_mask = X_eval_all.index >= holdout.index.min()
X_holdout_final = X_eval_all.loc[holdout_mask]
y_holdout_final = y_eval_all[holdout_mask]
y_naive_holdout_final = y_naive_eval_all[holdout_mask]

start_time = time.time()
preds_holdout = model_production.forecast(
    X_seq=X_holdout_final,
    y_seq=y_holdout_final,
    window=window,
    embargo=trainer.embargo,
    n_splits=trainer.n_splits,
)
preds_time = time.time() - start_time

df_preds = pd.DataFrame({
    "date": X_holdout_final.index,
    "y_true": y_holdout_final,
    "y_naive": y_naive_holdout_final,
    "y_pred": preds_holdout,
})

output_path = RESULTS_PATH / "lgbm_holdout_predictions_120.csv"
df_preds.to_csv(output_path, index=False)

En el diagnóstico de los residuales estandarizados no solapados sobre el conjunto *holdout*, tanto las pruebas de Ljung-Box como las pruebas ARCH-LM resultan estadísticamente significativas en todos los rezagos evaluados tras aplicar la corrección de Holm ($p_{\text{adj}}<.001$), con excepción de ARCH-LM en el rezago 30, cuyo valor ajustado es $p_{\text{adj}}=0.004$. A diferencia del cierre de entrenamiento, donde se identificó dependencia serial significativa pero no efectos ARCH, el desempeño fuera de muestra exhibe tanto autocorrelación persistente como agrupamiento de volatilidad desde los primeros rezagos evaluados.

Esta dinámica residual podría estar relacionada con los desplazamientos de nivel y las perturbaciones operativas no observadas durante el segundo semestre de 2025, capaces de generar rachas consecutivas de subestimación o sobreestimación entre bloques de reentrenamiento. Debido a que el ensamble estima la media condicional sin modelar explícitamente la estructura autorregresiva ni la heterocedasticidad condicional del error, los resultados sugieren que persisten patrones de dependencia temporal y variabilidad residual no capturados completamente por el modelo en escenarios de demanda no estacionaria.

In [20]:
residuals_test = (
    df_preds["y_true"].to_numpy() - df_preds["y_pred"].to_numpy()
)

block_size = 1000
horizon_slots = 8
residuals_block = residuals_test[-block_size::horizon_slots]
std_residuals_block = (
    (residuals_block - residuals_block.mean())
    / residuals_block.std(ddof=1)
)

diag_test = tt.run_model_diagnostics(
    std_residuals_block,
    lags=(10, 20, 30),
)

tt.display_styled_df(
    df=diag_test,
    p_value_cols=["p-valor", "p-ajustado"],
    align_left_first=True,
)

Prueba,Estadístico,p-valor,p-ajustado (holm),Conclusión (α=0.05)
Ljung-Box (Lag 10),71.494,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),85.236,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),90.148,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),56.174,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),57.736,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),54.833,.004,.004,Efectos ARCH Significativos


De acuerdo con los criterios establecidos en el protocolo experimental, no se observa evidencia de sobreajuste (*overfitting*) severo ni de subajuste (*underfitting*) en el modelo LightGBM-Tweedie. La función de pérdida principal (*Tweedie Deviance*) presenta una reducción relativa del $59.16\%$ entre entrenamiento y prueba, pasando de $0.2453$ a $0.1002$, lo que respalda una capacidad de generalización favorable fuera de muestra. Este comportamiento es consistente con las reducciones registradas en las métricas `MAE` ($0.05\%$), `RMSE` ($4.18\%$), `MSE` ($8.19\%$), `Pinball` ($0.05\%$) y `SMAPE` ($1.83\%$), evidenciando una estabilidad general de la precisión predictiva, con mejoras moderadas en los errores cuadráticos durante el período de prueba.

Las principales discrepancias se presentan en el `MASE`, que aumenta un $21.77\%$ (de $0.5293$ a $0.6445$), y en el `Bias`, que cambia de $-48.63$ a $85.69$ pasajeros, indicando una transición de subestimación promedio durante el entrenamiento a sobreestimación fuera de muestra. Esta inversión del sesgo sugiere diferencias en los patrones de demanda entre ambos períodos, posiblemente asociadas con los cambios de régimen identificados en el EDA de la ruta El Silencio. Por su parte, el incremento del `MASE` indica una reducción de la ventaja relativa frente al pronóstico ingenuo estacional, aunque sus valores inferiores a uno confirman que el modelo conserva un desempeño superior al referente utilizado en ambos conjuntos.

In [24]:
df_preds = pd.read_csv(RESULTS_PATH / "lgbm_holdout_predictions_120.csv")

X_train_full, y_train_full, y_naive_train_full, _ = trainer._build_dataset(train)
preds_train_final = model_production.predict_in_sample(X_train_full.iloc[-window:])

ts.display_train_test_comparison(
    y_train=y_train_full[-window:],
    preds_train=preds_train_final,
    y_naive_train=y_naive_train_full[-window:],
    y_test=df_preds["y_true"].to_numpy(),
    preds_test=df_preds["y_pred"].to_numpy(),
    y_naive_test=df_preds["y_naive"].to_numpy(),
    p=1.950592166020216,
    error_relativo=True,
)

Métrica,Entrenamiento (In-Sample),Prueba (Out-of-Sample),Error Relativo (%)
Tweedie Deviance,0.2453,0.1002,-59.1644%
MAE,245.6584,245.5272,-0.0534%
RMSE,398.4127,381.7453,-4.1834%
MSE,158732.6632,145729.4844,-8.1919%
Bias,-48.6343,85.6914,+276.1956%
SMAPE,0.1143,0.1122,-1.8265%
MASE,0.5293,0.6445,+21.7690%
Pinball (τ=0.50),122.8292,122.7636,-0.0534%


La inspección visual de las trayectorias predichas frente a la demanda realizada en el conjunto *holdout* muestra que el modelo reproduce con alta fidelidad la envolvente diaria, la estacionalidad semanal y el perfil multimodal intradía a lo largo del segundo semestre. A escala agregada diaria (panel superior), el ajuste sigue con precisión la demanda operativa en días hábiles y corrige el sesgo a la baja observado en la ventana de entrenamiento mediante el reentrenamiento deslizante, presentando únicamente episodios puntuales de sobreestimación asociados a caídas atípicas de la demanda, particularmente durante junio, julio y octubre, así como en los cierres operativos de final de diciembre.

A escala de ventanas acumuladas de 120 minutos (paneles inferiores, correspondientes a la primera semana de septiembre y al 15 de octubre de 2025), persiste una ligera subestimación en algunos picos matutinos de septiembre (con predicciones cercanas a $4600\text{--}5100$ pasajeros frente a máximos reales de $5000\text{--}5200$), aunque de magnitud sustancialmente menor que la registrada durante el entrenamiento. Por su parte, el 15 de octubre las predicciones reproducen con precisión la cúspide de la hora pico matutina (alrededor de las 05:00–05:30), alcanzando valores cercanos a $4900$ pasajeros frente a máximos reales de $5000$. Este comportamiento responde al efecto de suavizado propio de la estimación de la media condicional mediante árboles de decisión frente a fluctuaciones de alta frecuencia, sin comprometer la captura de la meseta de media mañana ni el repunte vespertino.

In [25]:
ts.plot_temporal_fit_panel(
    dates=pd.to_datetime(df_preds["date"]),
    y_true=df_preds["y_true"].to_numpy(),
    y_pred=df_preds["y_pred"].to_numpy(),
    zoom_left=("2025-09-01", "2025-09-07"),
    zoom_right=("2025-10-15", "2025-10-15"),
    text_left="Septiembre 2025",
    text_right="15 Octubre 2025",
    title="Demanda acumulada de pasajeros Holdout 2025 (120 min)",
    xlab_left="Día del mes",
    xlab_right="Hora del día",
    ylab_top="Pasajeros acumulados",
    ylab_zoom="Pasajeros por ventana (120 min)",
    zoom_ylim_pad=(0.10, 0.12),
    hspace=0.1,
    wspace=0.05,
)